# VGG（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，VGGを用いてCIFAR100データセットの100クラス物体認識を行い，小さな畳み込みカーネルを積み重ねることで深いネットワークを構築するというVGGの設計思想について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## VGG
VGGは，2014年のILSVRCで2位となったモデルであり，シンプルな構造ながら層を深くすることで高い認識精度を達成できることを示した代表的なモデルです．

VGGの最大の特徴は，畳み込み層のカーネルサイズを**すべて3×3に統一**した点にあります．例えば，5×5のカーネルを1つ使う代わりに3×3のカーネルを2つ重ねると，受容野（入力画像上で1つの出力に影響する範囲）は同じ5×5になりますが，活性化関数（ReLU）を通す回数が増えるためより複雑な非線形変換を表現でき，かつパラメータ数も削減できます（5×5×1個 = 25に対し，3×3×2個 = 18）．同様に，7×7のカーネル1つは3×3のカーネル3つと同じ受容野になります．このように小さなカーネルを積み重ねる設計が，VGGの核となる工夫です．

ネットワーク全体は，「3×3畳み込み（+ Batch Normalization + ReLU）を数層重ねてから2×2のMax Poolingで空間サイズを半分にする」というブロックを繰り返す，非常に規則的な構造になっています．ブロックを重ねるごとにチャンネル数を64→128→256→512→512と倍増させていき，最後に全結合層でクラス分類を行います．1ブロックあたりの畳み込み層数を変えることで，VGG11・VGG13・VGG16・VGG19といった層数の異なるバリエーションが定義されています．

なお，原論文のVGGにはBatch Normalizationは使われていません（Batch Normalizationの提案はVGGより後の2015年）が，本ノートブックでは学習を安定・高速化するために追加したBatch Normalization付きの構成（VGG-BN）を実装します．

### CIFAR100への入力サイズの適応
VGGには，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR10/100を対象とした「CIFAR版」があります．畳み込み部分（3×3畳み込みの積み重ねと5回のMax Pooling）の構造自体は共通ですが，Poolingを経た後の特徴マップサイズがImageNet版の7×7に対しCIFAR版では1×1まで縮小されるため，全結合層のユニット数を大きく減らす必要があります．本ノートブックでは，CIFAR100向けに全結合層を縮小したCIFAR版のVGGを実装します（詳細は本ノートブック末尾の「ImageNet版VGGとCIFAR版VGGの違い」を参照）．

In [ ]:
# 'M'はMax Poolingを表す．数値はその位置の畳み込み層の出力チャンネル数．
VGG_CONFIGS = {
    'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}


class VGG(nn.Module):
    def __init__(self, config='VGG16', n_class=100):
        super().__init__()
        self.features = self._make_layers(VGG_CONFIGS[config])
        # 32x32の入力はMax Poolingを5回経ることで1x1まで縮小されるため，
        # ImageNet版のような4096ユニットの全結合層は過剰であり，小さな全結合層で十分．
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_class),
        )

    def _make_layers(self, config):
        layers = []
        in_channels = 3
        for v in config:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                layers += [nn.Conv2d(in_channels, v, kernel_size=3, padding=1),
                           nn.BatchNorm2d(v),
                           nn.ReLU(inplace=True)]
                in_channels = v
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

## ネットワークの作成
定義したネットワークを作成します．`config`でVGGの層数（VGG11 / VGG13 / VGG16 / VGG19）を指定します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
# VGGの構成を指定 (VGG11 / VGG13 / VGG16 / VGG19)
config = 'VGG16'

model = VGG(config=config, n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版VGGとCIFAR版VGGの違い
このノートブックで実装したVGGは，32×32ピクセルのCIFAR100に合わせて構造を変更したものです．原論文のImageNet版VGGとは，主に次の点が異なります．

| | ImageNet版（原論文） | CIFAR版（本ノートブック） |
|---|---|---|
| 入力画像サイズ | 224×224 | 32×32 |
| 畳み込みのカーネルサイズ | 3×3（変更なし） | 3×3 |
| Max Poolingの回数 | 5回 | 5回（変更なし） |
| Poolingを経た後の特徴マップサイズ | 7×7×512 | 1×1×512 |
| 全結合層のユニット数 | 4096-4096-1000 | 512-100 |
| Batch Normalization | なし（原論文．別途VGG-BN版が存在） | あり（本ノートブックで使用） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

畳み込み部分の構造（3×3畳み込みの積み重ね，5回のMax Pooling）自体はImageNet版と変わりませんが，CIFAR100では入力が32×32と小さいため，5回のPoolingを経た時点で特徴マップは1×1にまで縮小されます．そのため，7×7×512（=25088次元）を4096ユニットの全結合層で処理するImageNet版の全結合層をそのまま用いると，1×1×512（=512次元）の入力に対して著しく過剰なパラメータ数となってしまいます．本ノートブックでは，この点を踏まえて全結合層を512-100という小さな構成に縮小しています．

## 課題

1. `config`を`VGG11`に変更し，`VGG16`との層数・パラメータ数・認識精度の違いを比較しましょう．

2. Batch Normalizationを取り除いたネットワークを作成し，学習の収束の速さや安定性，最終的な認識精度を比較しましょう．

3. 全結合層のユニット数を変更し，パラメータ数と認識精度の関係を確認しましょう．